# Prompt templates

In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

We can use Python built-in templates with substitutions, but it makes it difficult to substitute in real-time:

In [2]:
job_description = """
Python Developer

3 Years experience required with SQL, Bigquery. 
"""

In [4]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

prompt_template = (
    "Given a job description, decide whether it suites a junior Java developer."
    "\nJOB DESCRIPTION:\n{job_description}\n"
)
result = llm.invoke(prompt_template.format(job_description=job_description))
print(result.content)

No, this job description does **not** suit a junior Java developer.

Here's why:

1.  **Primary Language Mismatch:** The job explicitly states "Python Developer" and lists Python as the core skill. A junior Java developer's primary expertise is in Java.
2.  **Experience Level:** "3 Years experience required" is generally beyond what's considered a "junior" role, which typically implies 0-2 years of experience. This role sounds more mid-level.


Instead, we can use built-in templates provided by LangChain:

In [5]:
from langchain_core.prompts import PromptTemplate

lc_prompt_template = PromptTemplate.from_template(prompt_template)
lc_prompt_template.invoke({"job_description": "fake_jd"})

StringPromptValue(text='Given a job description, decide whether it suites a junior Java developer.\nJOB DESCRIPTION:\nfake_jd\n')

In [6]:
from langchain_core.output_parsers import StrOutputParser

lc_prompt_template = PromptTemplate.from_template(prompt_template)
chain = lc_prompt_template | llm | StrOutputParser()
chain.invoke({"job_description": job_description})

'No, this job description does **not** suit a junior Java developer.\n\nHere\'s why:\n\n1.  **Primary Language Mismatch:** The job explicitly states "Python Developer." A junior Java developer\'s primary skill set is Java, not Python.\n2.  **Experience Level Mismatch:** "3 Years experience required" is typically for a mid-level role, not a junior one (which usually implies 0-2 years of professional experience).\n3.  **Specific Skills:** While SQL is common across many development roles, BigQuery is more specialized and often associated with data engineering or Python-heavy data science roles.\n\nThis job is clearly for a mid-level Python developer with experience in data technologies.'

In [7]:
chain.invoke(job_description)

'No, this job description **does not suit a junior Java developer.**\n\nHere\'s why:\n\n1.  **Primary Language Mismatch:** The job explicitly asks for a "Python Developer," not a Java developer. While some developers might know both, the core requirement is Python.\n2.  **Experience Level Mismatch:** The job requires "3 Years experience." A "junior" role typically implies 0-2 years of experience. Even if a junior Java developer has 3 years of experience in Java, they would not have 3 years of experience as a Python developer unless they\'ve specifically pivoted their career.\n3.  **Specific Technologies:** While SQL is common, the explicit mention of "BigQuery" with 3 years of experience, combined with Python, points to a very specific skill set that a general junior Java developer is unlikely to possess.'

We can also use a special placeholder for messages:

In [12]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import SystemMessagePromptTemplate

msg_template = HumanMessagePromptTemplate.from_template(prompt_template)
msg_example = msg_template.format(job_description="fake_jd")

print(msg_example.content)
print()
chat_prompt_template = ChatPromptTemplate.from_messages(
    [SystemMessage(content="You are a helpful assistant"), msg_template]
)
chain = chat_prompt_template | llm | StrOutputParser()
chain.invoke({"job_description": job_description})

Given a job description, decide whether it suites a junior Java developer.
JOB DESCRIPTION:
fake_jd




'No, this job description **does not suit a junior Java developer.**\n\nHere\'s why:\n\n1.  **Primary Language:** The job title and core requirement is "Python Developer," not Java. A junior Java developer\'s primary skill set and experience would be in Java.\n2.  **Experience Level:** "3 Years experience required" is typically more in line with a mid-level developer, not a junior developer (who usually has 0-2 years of experience). Even if a junior Java developer had 3 years of experience, it would be in Java, not Python.\n3.  **Specific Technologies:** While SQL and BigQuery are valuable skills for many developers, they don\'t compensate for the lack of Python expertise required for this role, nor do they make it a Java-centric position.'

In [ ]:
# Using String Message Type is also allowed here
chat_prompt_template = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant."), ("human", prompt_template)]
)
chain = chat_prompt_template | llm | StrOutputParser()
result = chain.invoke({"job_description": job_description})
print(result)

No, this job description does **not** suit a junior Java developer.

Here's why:

1.  **Primary Language Mismatch:** The job title and core requirement is "Python Developer." A junior Java developer's primary skill set and focus is the Java programming language.
2.  **Experience Mismatch:** The role requires "3 Years experience" with Python (implied by the title) and data technologies like SQL and Bigquery. A junior Java developer would have 0-3 years of experience primarily in Java, not Python.

While a developer might have some exposure to multiple languages, this role is clearly looking for a Python specialist with established experience in that field.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("placeholder", "{history}"),
        # same as MessagesPlaceholder("history"),
        ("human", prompt_template),
    ]
)

In [17]:
chat_prompt_template.invoke("fake")

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Given a job description, decide whether it suites a junior Java developer.\nJOB DESCRIPTION:\nfake\n', additional_kwargs={}, response_metadata={})])

In [ ]:
len(
    chat_prompt_template.invoke(
        {"job_description": "fake 2", "history": [("human", "hi"), ("ai", "hi!")]}
    ).messages
)

4

In [19]:
examples = [{"question": "q1", "answer": "a1"}, {"question": "q2", "answer": "a2"}]
test_template = PromptTemplate.from_template("substituted a: {a}")

And we can use partial substitutions with templates:

In [20]:
system_template = PromptTemplate.from_template("a: {a} b: {b}")
system_template_part = system_template.partial(
    a="a"
)

print(system_template_part.invoke({"b": "b"}).text)

system_template_part.invoke({"b": "b"}).text == system_template_part.format(b="b")

a: a b: b


True

Concatenate the Prompt Templates

In [26]:
system_template_part1 = PromptTemplate.from_template("a: {a}")
system_template_part2 = PromptTemplate.from_template("b: {b}")

system_template = system_template_part1 + " " + system_template_part2
print(system_template.invoke({"a": "a", "b":"b"}).text)

a: a b: b


In [28]:
system_prompt_template = PromptTemplate.from_template("a: {a} b: {b}")
chat_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_template.template),
        ("human", "hi"),
        ("ai", "{c}")
    ]
)

messages = chat_prompt_template.invoke({"a": "a", "b": "b", "c": "c"}).messages
print(len(messages))
print(messages[0].content)

3
a: a b: b


In [32]:
system_prompt_template = PromptTemplate.from_template("a: {a} b: {b}")
chat_prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_prompt_template.template), ("human", "hi"), ("ai", "{c}")]
)

messages = chat_prompt_template.invoke({"a": "a", "b": "b", "c": "c"}).messages
print(len(messages))
print(messages[0].content)
print(messages[1].content)
print(messages[2].content)

3
a: a b: b
hi
c
